# MELA-260907 (PyTorch) on a Colab TPU via PyTorch/XLA
Runtime -> TPU. Upload `MELA-260907.zip` when asked. Steps: install torch_xla if missing, device check, forward equivalence XLA-vs-CPU, fallback (aten::) report, one training step timing at d=256 B=16 T=512.

In [ ]:
import os, sys, time
try:
    import torch_xla
except ImportError:
    !pip -q install torch~=2.8.0 torch_xla[tpu]~=2.8.0 -f https://storage.googleapis.com/libtpu-releases/index.html
import torch, torch_xla, torch_xla.core.xla_model as xm
import torch_xla.debug.metrics as met
print(torch.__version__, torch_xla.__version__)
dev = torch_xla.device(); print('device', dev)
from google.colab import files
if not os.path.exists('MELA-260907'):
    up = files.upload(); import zipfile; zipfile.ZipFile(list(up.keys())[0]).extractall('.')
sys.path.insert(0, 'MELA-260907')
from mela260907 import Config, MELALayer, LM
from mela260907.walk import walk_uniforms

In [ ]:
# 1) forward equivalence XLA vs CPU + fallback report
torch.manual_seed(0)
cfg = Config(d=64, T=256, chunk=64, recompute_walk=False, recompute_transport=False)
lay = MELALayer(cfg); h = torch.randn(2, 256, 64)
n_ev = len(range(cfg.k_event, 256, cfg.k_event))
u = walk_uniforms(n_ev, 2, cfg.n_walks, cfg.walk_len, seed=7, device='cpu')
with torch.no_grad():
    o_cpu = lay(h, u=u)
lay_x = MELALayer(cfg).to(dev); lay_x.load_state_dict(lay.state_dict())
met.clear_all()
with torch.no_grad():
    o_x = lay_x(h.to(dev), u=u.to(dev)); xm.mark_step()
print('XLA vs CPU rel', float((o_x.cpu() - o_cpu).abs().max() / o_cpu.abs().max()))
rep = met.metrics_report()
print('\n'.join(l for l in rep.splitlines() if 'aten::' in l) or 'no aten:: fallbacks')

In [ ]:
# 2) training-step time at d 256, B 16, T 512, 2 layers (XLA: first steps compile)
d, T, B, V, layers = 256, 512, 16, 65, 2
cfg = Config(d=d, T=T)
m = LM(V, cfg, layers).to(dev)
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
n_ev = len(range(cfg.k_event, T, cfg.k_event))
x = torch.randint(0, V, (B, T), device=dev)
def us_for(i):
    return [walk_uniforms(n_ev, B, cfg.n_walks, cfg.walk_len, seed=100 * i + b, device='cpu').to(dev) for b in range(layers)]
ts = []
for i in range(5):
    t0 = time.perf_counter()
    logits = m(x, us=us_for(i)); loss = torch.nn.functional.cross_entropy(logits.reshape(-1, V), x.reshape(-1))
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); xm.mark_step(); xm.wait_device_ops()
    ts.append(time.perf_counter() - t0); print('step', i, '%.2f s' % ts[-1], 'loss', float(loss))
print('XLA TPU step s (median of last 3):', sorted(ts[2:])[1])
print('\n'.join(l for l in met.metrics_report().splitlines() if 'aten::' in l or 'CompileTime' in l or 'ExecuteTime' in l)[:2000])